# 01 · Topomapas y montajes (validación de referencias)

**Rama `explore/topomap-refs`** · ejecución local con datos reales cacheados
(eegbci 64ch canónico, `standard_1005`, sujetos 1-2, 7 referencias).

Objetivos:
1. Cargar los datos reales y comprobar con **métricas correctas** que las
   7 referencias almacenadas en `data/processed/` son consistentes entre sí.
2. Reproducir **topomapas** estilo MNE (disco circular, medido negro /
   interpolado gris) sobre la malla esférica.
3. **Derivar montajes por interpolación esférica**: sacar la 10-20 (19) y la
   10-10 (42) reales como subconjuntos del canonical de 64 canales, y medir si
   la 10-20 interpolada desde la 10-10 (o desde todo el casco) reproduce la
   10-20 medida (VE y RMSE por canal).

Convención matricial: `X_ref = X @ M` (las matrices actúan por filas).

## 0. Entorno y configuración

In [1]:
import os, sys, json
import numpy as np

ROOT = os.getcwd()
if os.path.basename(ROOT) == "topomap_refs":
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
for p in (os.path.join(ROOT, "src"),):
    if p not in sys.path:
        sys.path.insert(0, p)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from eeg_transform.nb import load_experiment, multiconfig_data
from eeg_transform.config import REFERENCE_KINDS
from eeg_transform.references import build_reference_matrix, inter_reference_matrix
from eeg_transform.mapping import (select_subset, STANDARD_10_20_19,
                                   spherical_spline_matrix, scalp_grid_matrix)
from eeg_transform.experiments.multi import MONTAGE_10_10_39

OUT = os.path.join(ROOT, "runs", "topomap_refs")
FIG = os.path.join(OUT, "01_figs")
os.makedirs(FIG, exist_ok=True)

def ve(a, b):
    """Varianza explicada (adimensional, escala-invariante) de a frente a b."""
    a0 = a - a.mean(0, keepdims=True)
    b0 = b - b.mean(0, keepdims=True)
    return float(1 - ((a0 - b0) ** 2).sum() / (b0 ** 2).sum())

print("ROOT:", ROOT)
print("referencias:", REFERENCE_KINDS)

I0000 00:00:1790015642.412253   99345 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790015642.412693   99345 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790015642.445286   99345 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790015643.340667   99345 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790015643.340879   99345 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


ROOT: /home/aess/Proyectos/universal-eeg-transformer
referencias: ('unipolar', 'linked_mastoids', 'linked_ears', 'bipolar', 'car', 'rest', 'laplacian')


## 1. Datos reales cacheados (canonical eegbci 64ch, 7 refs)

In [2]:
cfg, ds = load_experiment("config/universal_refs.yaml")
data = multiconfig_data(cfg, ds)
mc = data["canonical"]
refs = mc.refs["test"]

print("configuraciones:", data.order)
print("canales:", ds.n_channels, "| radio posiciones (m):", float(np.median(np.linalg.norm(ds.ch_positions, axis=1))).__round__(5))
print("muestras train/val/test:", {k: len(v) for k, v in ds.split_idx.items()})
print("referencias:", list(refs.keys()))
print("leadfield:", mc.leadfield.shape, "| malla superficie:", np.shape(getattr(mc, "surface", None)))

configuraciones: ['canonical']
canales: 64 | radio posiciones (m): 0.11769
muestras train/val/test: {'train': 11642, 'val': 2910, 'test': 3638}
referencias: ['unipolar', 'linked_mastoids', 'linked_ears', 'bipolar', 'car', 'rest', 'laplacian']
leadfield: (64, 750) | malla superficie: (1024, 64)


## 2. ¿Son correctas las referencias aplicadas?

La métrica debe respetar las **unidades de cada operador**: car/rest/bipolar/
linked* son potenciales (µV); el **laplaciano (CSD) está en V/m²** (sus RMSE
crudos no son comparables en µV). Batería:

* **C1 · Recuperación del operador desde unipolar** con **VE** para las 7:
  `X_unipolar @ M_k` debe reproducir `refs[k]` (RMSE en µV solo para la
  familia de potenciales; la CSD exacta deja residuos de ~0.08 % con VE).
* **C2 · Canal de referencia nulo** en unipolar (`columna ref_index == 0`).
* **C3 · Laplaciano invariante a la re-referenciación**: operador `L` cuyas
  columnas suman ~0 (anula el modo común de la entrada) y que además recupera
  la CSD almacenada con VE alto.
* **C4 · Rutas cruzadas (7×7) con VE**: se comprueba `X_s @ M_{s→d} ≈ X_d`.
  ⚠️ Aquí se usa la composición **corregida** `pinv(T_s) @ T_d`
  (convención de filas `X @ M`); `inter_reference_matrix` del paquete devuelve
  `T_d @ pinv(T_s)` y **no reproduce** las señales (se documenta abajo).
* **C5 · Modo común (media de la referencia)**: mediana del promedio sobre
  canales *y* pico; los picos deben interpretarse como artefactos puntuales
  de EEG, no fuga sistemática.

In [3]:
C = ds.n_channels
kw = dict(unipolar_ref_index=mc.unipolar_ref_index, lead_field=mc.leadfield,
          rest_rcond=cfg.leadfield.rest_rcond, positions=mc.positions)
Xu = refs["unipolar"]
results = {"C1": {}, "C2": {}, "C3": {}, "C4": {}, "C5": {}, "bug_inter_reference": {}}

# C1: recuperación del operador desde unipolar
Ms = {k: build_reference_matrix(k, C, **kw) for k in REFERENCE_KINDS}
print("C1 · recuperación desde unipolar (X_u @ M_k vs refs[k]):")
for k in REFERENCE_KINDS:
    E = Xu @ Ms[k] - refs[k]
    v = ve(Xu @ Ms[k], refs[k])
    rmse_uV = float(np.sqrt((E ** 2).mean()) * 1e6)
    results["C1"][k] = {"VE": v, "RMSE_uV": rmse_uV if k != "laplacian" else None}
    rmse_s = f"| RMSE={rmse_uV:.3f} µV" if k != "laplacian" else "| RMSE n/a (V/m²)"
    print(f"  {k:<16s} VE={v:.6f} {rmse_s}  {'PASS' if v > 0.999 else 'REVISAR'}")

# C2: canal de referencia nulo
cz = float(np.abs(Xu[:, mc.unipolar_ref_index]).max()) * 1e6
results["C2"]["max_abs_Cz_uV"] = cz
print(f"C2 · |unipolar[:, ref=Cz]| max = {cz:.6f} µV  ({'PASS' if cz < 1e-6 else 'REVISAR'})")

# C3: laplaciano → anula modo común + recupera la CSD
L = Ms["laplacian"]
rc = np.random.default_rng(7).normal(size=(64, 1))
com = rc @ np.ones((1, C))
kill = float(np.abs(com @ L).max())
vLs = ve(Xu @ L, refs["laplacian"])
results["C3"] = {"max_abs_colsum": float(np.abs(L.sum(0)).max()),
                 "modo_comun_salida": kill, "VE_unipolar_csd": vLs,
                 "RMSE_csd": float(np.sqrt(((Xu @ L - refs["laplacian"]) ** 2).mean()))}
print("C3 · laplaciano:")
print(f"   |Σ_col L| max     = {results['C3']['max_abs_colsum']:.3e}  (esperado ~0: anula modo común de la entrada)")
print(f"   |modo común @ L|  = {kill:.3e}  (debe ser ~0)")
print(f"   VE(X_u @ L, CSD)  = {vLs:.6f}   RMSE(CSD) = {results['C3']['RMSE_csd']:.3e}")
print("   ->", "PASS" if (kill < 1e-3 and vLs > 0.999) else "REVISAR")

C1 · recuperación desde unipolar (X_u @ M_k vs refs[k]):
  unipolar         VE=1.000000 | RMSE=0.000 µV  PASS
  linked_mastoids  VE=1.000000 | RMSE=0.015 µV  PASS
  linked_ears      VE=1.000000 | RMSE=0.019 µV  PASS
  bipolar          VE=1.000000 | RMSE=0.000 µV  PASS
  car              VE=1.000000 | RMSE=0.000 µV  PASS
  rest             VE=1.000000 | RMSE=0.000 µV  PASS
  laplacian        VE=0.999646 | RMSE n/a (V/m²)  PASS
C2 · |unipolar[:, ref=Cz]| max = 0.000000 µV  (PASS)
C3 · laplaciano:
   |Σ_col L| max     = 2.337e-08  (esperado ~0: anula modo común de la entrada)
   |modo común @ L|  = 5.881e-08  (debe ser ~0)
   VE(X_u @ L, CSD)  = 0.999646   RMSE(CSD) = 8.216e-04
   -> PASS


In [4]:
# C4: rutas cruzadas, con la composición CORREGIDA para la convención de filas
K = tuple(REFERENCE_KINDS)
def cross_matrix(t_src, t_dst):
    """M_{s->d} en convención X @ M: de la referencia s a la d vía espacio crudo."""
    return np.linalg.pinv(t_src, rcond=1e-10) @ t_dst

VE = np.zeros((len(K), len(K)))
for i, s in enumerate(K):
    for j, d in enumerate(K):
        VE[i, j] = ve(refs[s] @ cross_matrix(Ms[s], Ms[d]), refs[d])
results["C4"] = {"median": float(np.median(VE)), "min": float(VE.min()),
                 "worst": [K[VE.argmin() // len(K)], K[VE.argmin() % len(K)]]}
print("C4 · VE rutas cruzadas (pinv(T_s) @ T_d): mediana=%.6f  mín=%.4f (peor %s)"
      % (results["C4"]["median"], results["C4"]["min"], results["C4"]["worst"]))
for i, s in enumerate(K):
    for j, d in enumerate(K):
        if VE[i, j] < 0.999:
            print(f"      {s}->{d}: {VE[i, j]:.4f}")

# Documentar el bug del paquete (mismo objetivo, otra composición)
VE_bug = np.zeros((len(K), len(K)))
for i, s in enumerate(K):
    for j, d in enumerate(K):
        VE_bug[i, j] = ve(refs[s] @ inter_reference_matrix(s, d, C, **kw), refs[d])
results["bug_inter_reference"] = {"median_package_inter_reference": float(np.median(VE_bug)),
                                  "min_package": float(VE_bug.min()),
                                  "composicion_correcta": "pinv(T_s) @ T_d",
                                  "composicion_paquete": "T_d @ pinv(T_s)"}
print("BUG (paquete) · inter_reference_matrix = T_d @ pinv(T_s) -> mediana VE = "
      "%.4f (mín %.2f). La corregida da mediana 1.0; fix propuesto en src/"
      % (results["bug_inter_reference"]["median_package_inter_reference"],
         results["bug_inter_reference"]["min_package"]))

fig, ax = plt.subplots(figsize=(7.4, 6))
im = ax.imshow(VE, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(K))); ax.set_xticklabels(K, rotation=45, ha="right")
ax.set_yticks(range(len(K))); ax.set_yticklabels(K)
for i in range(len(K)):
    for j in range(len(K)):
        ax.text(j, i, f"{VE[i,j]:.3f}", ha="center", va="center", fontsize=8,
                color="w" if VE[i, j] < 0.8 else "k")
ax.set_title("C4 · VE rutas cruzadas corregidas (49)")
fig.colorbar(im, ax=ax, shrink=0.85)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "routes_ve.png"), dpi=110); plt.close(fig)

C4 · VE rutas cruzadas (pinv(T_s) @ T_d): mediana=1.000000  mín=0.9309 (peor ['rest', 'laplacian'])
      rest->unipolar: 0.9804
      rest->linked_mastoids: 0.9979
      rest->linked_ears: 0.9985
      rest->bipolar: 0.9807
      rest->car: 0.9962
      rest->laplacian: 0.9309


BUG (paquete) · inter_reference_matrix = T_d @ pinv(T_s) -> mediana VE = 0.5670 (mín -492.93). La corregida da mediana 1.0; fix propuesto en src/


In [5]:
# C5: modo común de cada referencia (mediana robusta + pico de artefacto)
print("C5 · media sobre canales por instante (mediana | pico):")
for k in K:
    cm = refs[k].mean(1)
    med = float(np.median(np.abs(cm)))
    if k == "laplacian":
        med, peak, u = med, float(np.abs(cm).max()), "V/m²"
    else:
        med, peak, u = med * 1e6, float(np.abs(cm).max()) * 1e6, "µV"
    results["C5"][k] = {"mediana_abs": med, "pico_abs": peak, "unidades": u}
    print(f"  {k:<16s} mediana={med:9.3f} {u}  pico={peak:10.3f} {u}")

print("Nota: los picos corresponden a muestras individuales de artefacto (EEG);",
      "la mediana (~µV) es el modo común real y debe ser ~0 para car/bipolar/CSD.")

C5 · media sobre canales por instante (mediana | pico):
  unipolar         mediana=    8.984 µV  pico=    55.585 µV
  linked_mastoids  mediana=   10.288 µV  pico=    83.251 µV
  linked_ears      mediana=   15.058 µV  pico=   105.328 µV
  bipolar          mediana=    0.000 µV  pico=     0.000 µV
  car              mediana=    0.000 µV  pico=     0.000 µV
  rest             mediana=  109.834 µV  pico=  1155.358 µV
  laplacian        mediana=    0.003 V/m²  pico=     0.019 V/m²
Nota: los picos corresponden a muestras individuales de artefacto (EEG); la mediana (~µV) es el modo común real y debe ser ~0 para car/bipolar/CSD.


### Conclusión de la validación

* `C1` recupera las 7 referencias desde unipolar con **VE ≥ 0.9996** (las 6 de
  potencial exactas; la CSD deja un residuo de ~0.04 % en sus unidades).
* `C2`: el canal de referencia (Cz) del unipolar es **0 exacto**.
* `C3`: la CSD anula el modo común de la entrada (operador invariante a la
  referencia); se recupera la CSD almacenada con VE alto.
* `C4`: con la composición **corregida** `pinv(T_s) @ T_d`, las 49 rutas
  cruzadas tienen **VE mediana 1.0**; las únicas bajas pasan por REST
  (rank 58 de 63) → pérdida de los grados de libertad que REST no codifica.
* ⚠️ **Hallazgo**: `inter_reference_matrix` del paquete (`T_d @ pinv(T_s)`)
  **no funciona** con la convención `X @ M` (VE mediana 0.57). Es un helper de
  validación; el fix pertenece a la rama madre (`src/`), fuera de alcance aquí.
* `C5`: modo común ~µV (mediana) con picos de artefacto puntuales.

⇒ Las 7 referencias de `dataset_block_1_2.npz` son **correctas y
consistentes**; queda el camino listo para `02_referencia_average_all` y
`03_modelo_montaje_referencia`.

## 3. Montajes por interpolación esférica (10-10 y 10-20)

In [6]:
n1020, p1020, i1020 = select_subset(ds.ch_names, ds.ch_positions, keep=STANDARD_10_20_19)
n1010, p1010, i1010 = select_subset(ds.ch_names, ds.ch_positions, keep=MONTAGE_10_10_39)
print("10-20 (canales reales):", len(n1020), n1020)
print("10-10 (canales reales):", len(n1010))
print("10-20 ⊆ 10-10:", set(n1020) <= set(n1010), "| 10-10 ⊆ canonical:", set(n1010) <= set(ds.ch_names))

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(p1010[:, 0], p1010[:, 1], s=32, c="tab:orange", label=f"10-10 ({len(n1010)})")
ax.scatter(p1020[:, 0], p1020[:, 1], s=60, facecolors="none", edgecolors="k", linewidths=1.5, label=f"10-20 ({len(n1020)})")
for (x, y, _), nm in zip(p1010, n1010):
    ax.annotate(nm, (x, y), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_aspect("equal"); ax.set_title("Posiciones MNE standard_1005 (subconjuntos reales)")
ax.legend(loc="upper right", fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "montage_positions.png"), dpi=110); plt.close(fig)

10-20 (canales reales): 19 ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T7', 'T8', 'P7', 'P8', 'Fz', 'Cz', 'Pz']
10-10 (canales reales): 42
10-20 ⊆ 10-10: True | 10-10 ⊆ canonical: True


In [7]:
_GRID_CACHE = {}
def grid(positions, grid_px=48):
    key = (id(positions), grid_px)
    if key not in _GRID_CACHE:
        _GRID_CACHE[key] = scalp_grid_matrix(positions, grid_px=grid_px)
    return _GRID_CACHE[key]

def topomap(Xt, positions, grid_px=48, ax=None, title="", markers=None, vlim=None, cmap="RdBu_r"):
    M, valid, _ = grid(positions, grid_px)
    img = (np.asarray(Xt).reshape(1, -1) @ M.T).reshape(grid_px, grid_px)
    img = np.where(valid.reshape(grid_px, grid_px), img, np.nan)
    if ax is None:
        ax = plt.gca()
    v = float(np.nanmax(np.abs(img))) if vlim is None else vlim
    im = ax.imshow(img, cmap=cmap, vmin=-v, vmax=v, origin="lower", interpolation="bilinear")
    if markers is not None:
        r_ax = np.linalg.norm(markers, axis=1, keepdims=True)
        nrm = markers / np.maximum(r_ax, 1e-12)
        cc = grid_px / 2 + (grid_px / 2 - 6) * nrm
        ax.scatter(cc[:, 1], cc[:, 0], s=8, color="k", alpha=0.7)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=9)
    return im

t = int(np.argmax(np.var(refs["unipolar"], axis=1)[:400]))
fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.4))
for ax, k in zip(axes, ["unipolar", "car", "laplacian"]):
    topomap(refs[k][t], ds.ch_positions, ax=ax, title=f"{k} @ t={t}", markers=p1020)
    fig.colorbar(ax.images[0], ax=ax, shrink=0.8, pad=0.02)
fig.suptitle("Topomapas canonical 64ch · disco circular (marcas = 10-20 medida)", y=1.02)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "topomaps_canonical.png"), dpi=110); plt.close(fig)
print("instante t =", t)

instante t = 7


## 3.1 Derivación de montaje: métricas

`spherical_spline_matrix(src, dst)` interpola entre distribuciones (*gauge* de
suma nula). En `test`, para dos referencias (unipolar y CAR):

* `canonical(64) -> 10-20(19)` y `10-10(42) -> 10-20(19)`: VE y RMSE (µV) del
  canal derivado vs el canal real medido.
* Canal `Cz` del unipolar es ~0 por definición (canal de referencia): se
  excluye del VE por canal (en CAR no hay canal degenerado).
* Round-trip `10-10 -> 10-20 -> 10-10` (VE).

In [8]:
# rutas de derivación
for refname in ["unipolar", "car"]:
    Xs = refs[refname]
    real = Xs[:, i1020]
    sm = cfg.mapping.smoothness
    P_c20 = spherical_spline_matrix(np.asarray(ds.ch_positions), np.asarray(p1020), smoothness=sm)
    P_1020 = spherical_spline_matrix(np.asarray(p1010), np.asarray(p1020), smoothness=sm)
    P_1020_10 = spherical_spline_matrix(np.asarray(p1020), np.asarray(p1010), smoothness=sm)
    if refname == "unipolar":
        global_metrics = {"derivacion": {}, "roundtrip": None}
    for name, P, src in [("canonical->10-20", P_c20, Xs), ("10-10->10-20", P_1020, Xs[:, i1010])]:
        der = src @ P
        per_ch = np.array([ve(der[:, j], real[:, j]) for j in range(len(n1020))])
        keep = [j for j in range(len(n1020)) if n1020[j] != "Cz"]
        d = {"ref": refname, "VE": ve(der, real),
             "RMSE_uV": float(np.sqrt(((der - real) ** 2).mean()) * 1e6),
             "VE_mediana_canal_sinCz": float(np.median(per_ch[keep])),
             "VE_min_canal_sinCz": float(per_ch[keep].min()),
             "canal_peor_sinCz": str(n1020[int(keep[per_ch[keep].argmin()])])}
        global_metrics["derivacion"][f"{refname}|{name}"] = d
        print(f"{refname:9s} {name:22s} VE={d['VE']:.4f} RMSE={d['RMSE_uV']:.3f} µV  "
              f"VE/canal(sin Cz): med={d['VE_mediana_canal_sinCz']:.4f} min={d['VE_min_canal_sinCz']:.4f} ({d['canal_peor_sinCz']})")
    # round-trip 10-10 -> 10-20 -> 10-10
    X_rt = Xs[:, i1010] @ P_1020 @ P_1020_10
    rt = ve(X_rt, Xs[:, i1010])
    if refname == "unipolar":
        global_metrics["roundtrip"] = {"ref": refname, "VE": rt, "RMSE_uV": float(np.sqrt(((X_rt - Xs[:, i1010]) ** 2).mean()) * 1e6)}
        print(f"round-trip 10-10 -> 10-20 -> 10-10 (unipolar)  VE={rt:.4f}")

# trazado de referencia (canales C3 y Pz de la 10-20, unipolar)
Xs = refs["unipolar"]; real = Xs[:, i1020]
sm = cfg.mapping.smoothness
P_c20 = spherical_spline_matrix(np.asarray(ds.ch_positions), np.asarray(p1020), smoothness=sm)
P_1020 = spherical_spline_matrix(np.asarray(p1010), np.asarray(p1020), smoothness=sm)
fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.4))
tt = np.arange(600)
for j, ax in zip([n1020.index("C3"), n1020.index("Pz")], axes):
    ax.plot(tt, real[:600, j] * 1e6, "k", lw=1, label="10-20 real")
    ax.plot(tt, (Xs[:, i1010][:600] @ P_1020)[:, j] * 1e6, "tab:red", lw=0.8, alpha=0.85, label="derivada 10-10→10-20")
    ax.plot(tt, (Xs[:600] @ P_c20)[:, j] * 1e6, "tab:orange", lw=0.8, alpha=0.85, label="derivada 64→10-20")
    ax.set_title(f"Canal {n1020[j]} · 10-20 derivada vs medida (µV)")
    ax.legend(fontsize=7); ax.grid(alpha=0.3); ax.set_xlabel("muestras (test)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "montage_derivation.png"), dpi=110); plt.close(fig)

unipolar  canonical->10-20       VE=0.9640 RMSE=4.874 µV  VE/canal(sin Cz): med=0.9708 min=0.8951 (F3)
unipolar  10-10->10-20           VE=0.9800 RMSE=3.630 µV  VE/canal(sin Cz): med=0.9805 min=0.9092 (F3)
round-trip 10-10 -> 10-20 -> 10-10 (unipolar)  VE=0.9429
car       canonical->10-20       VE=0.9422 RMSE=4.874 µV  VE/canal(sin Cz): med=0.9563 min=0.7677 (F3)
car       10-10->10-20           VE=0.9679 RMSE=3.630 µV  VE/canal(sin Cz): med=0.9730 min=0.7988 (F3)


/tmp/ipykernel_99345/1691514406.py:31: RuntimeWarning: divide by zero encountered in scalar divide
  return float(1 - ((a0 - b0) ** 2).sum() / (b0 ** 2).sum())


In [9]:
results["montage_derivation"] = global_metrics["derivacion"]
results["roundtrip"] = global_metrics["roundtrip"]
results["splits"] = {k: int(len(v)) for k, v in ds.split_idx.items()}
results["n_channels"] = int(C)
results["finding_inter_reference_matrix"] = (
    "inter_reference_matrix devuelve T_d @ pinv(T_s); con la convención X @ M la "
    "composición correcta es pinv(T_s) @ T_d. Bajo la implementación actual las "
    "rutas cruzadas NO se reproducen (VE mediana %.4f); corregida, mediana 1.0. "
    "Fix propuesto en las ramas de desarrollo (src/)."
    % results["bug_inter_reference"]["median_package_inter_reference"])
with open(os.path.join(OUT, "01_metrics.json"), "w") as fh:
    json.dump(results, fh, indent=2, default=float)

refs_ok = all(v["VE"] > 0.999 for v in results["C1"].values())
c4_ok = results["C4"]["median"] > 0.999
print("métricas ->", os.path.join(OUT, "01_metrics.json"))
print()
print("RESUMEN FINAL")
print("  C1 referencias consistentes desde unipolar:", refs_ok)
print("  C2 canal Cz nulo:", cz < 1e-6)
print("  C3 CSD invariante a la referencia:", kill < 1e-3 and vLs > 0.999)
print("  C4 rutas cruzadas VE mediana=%.6f ->" % results["C4"]["median"], "PASS" if c4_ok else "REVISAR")
d = results["montage_derivation"]["unipolar|10-10->10-20"]
print("  derivación 10-10 -> 10-20  VE=%.4f RMSE=%.3f µV" % (d["VE"], d["RMSE_uV"]))
d = results["montage_derivation"]["unipolar|canonical->10-20"]
print("  derivación canonical -> 10-20 VE=%.4f RMSE=%.3f µV" % (d["VE"], d["RMSE_uV"]))
print("  round-trip 10-10<->10-20   VE=%.4f" % results["roundtrip"]["VE"])

métricas -> /home/aess/Proyectos/universal-eeg-transformer/runs/topomap_refs/01_metrics.json

RESUMEN FINAL
  C1 referencias consistentes desde unipolar: True
  C2 canal Cz nulo: True
  C3 CSD invariante a la referencia: True
  C4 rutas cruzadas VE mediana=1.000000 -> PASS
  derivación 10-10 -> 10-20  VE=0.9800 RMSE=3.630 µV
  derivación canonical -> 10-20 VE=0.9640 RMSE=4.874 µV
  round-trip 10-10<->10-20   VE=0.9429
